In [0]:
select * from nctracs.vascular_pvl_measurements limit 10

In [0]:

--13k rows are null for anatomy -- not an issue at this point
--on pressure readings if there are 2 numbers, safe to assume 1st is always right laterality and 2nd is left
--on pressure readings if there are 4 numbers, safe to assume the first 2 readings are right and the 2nd two are left

create or replace temp view pvl_1 as
select deid_person_id, proc_start_date, resulting_lab_name, description, narrative, 

CASE WHEN narrative REGEXP '\\d+\\.\\d+' THEN '1' else 0 end as float_check,
 case when lower(narrative) like '%other%' then 'other' else 'measurement' end as type,

   REGEXP_EXTRACT(lower(narrative), r'right\s+(\S+)', 1) AS right,
   REGEXP_EXTRACT(lower(narrative), r'left\s+(\S+)', 1) AS left,

  case when lower(REGEXP_EXTRACT(lower(narrative), r'right\s+(\S+)', 1)) like '%unobtain%' then 999 end as rt_unobtainable,
  case when lower(REGEXP_EXTRACT(lower(narrative), r'left\s+(\S+)', 1)) like '%unobtain%' then 999 end as lt_unobtainable,

case when lower(narrative) like '%ankle%' or lower(narrative) like '%abi%' 
        or lower(narrative) like '%tibial%' or lower(narrative) like '%peroneal%' 
        or lower(narrative) like '%dorsalis%' or lower(narrative) like '%pta%'
        or lower(narrative) like '%ata%' or lower(narrative) like '%dp%' then 'ankle' 
     when lower(narrative) like '%brachial%' then 'arm'
     when lower(narrative) like '%toe%' or lower(narrative) like '%digit%' then 'toe'
         end as anatomy, 

  case when lower(narrative) like '%tibial%'  then 'tibial'
       when lower(narrative) like '%peroneal%' then 'peroneal'
       when lower(narrative) like '%dorsalis%' then 'dorsalis'
       when lower(narrative) like '%abi%' then 'abi'
         end as specific_anatomy, 

case when description like '%RIGHT%' then 'right'
     when description like '%LEFT%'  then 'left'
     when description like '%BILAT%' then 'bilateral' end as desc_laterality,   

case when lower(narrative) like '%unobtainable%' then 1 else 0 end as unobtainable, 

case when lower(narrative) like '%right%' or lower(narrative) like '%rt%' 
     or narrative like '%R %' then 'right' end as narr_right,

case when lower(narrative) like '%left%' or lower(narrative) like '%lt%' 
     or narrative like '%L %' then 'left' end as narr_left,     
  
   regexp_extract_all(narrative, '-?\\d*\\.\\d+', 0) AS extracted_float,
   size(regexp_extract_all(narrative, '-?\\d*\\.\\d+', 0)) as array_length,
   element_at(extracted_float,1) as value1,
   element_at(extracted_float,2) as value2,
   element_at(extracted_float,3) as value3,
   element_at(extracted_float,4) as value4
  --  REGEXP_EXTRACT_all(narrative, '-?\\d+(\\.\\d+)?', 0) AS extracted_number3,
from nctracs.vascular_pvl_measurements


In [0]:
select * from pvl_1 limit 100

In [0]:
--create dataset for unobtainable measurements
--map the pvl value to 999 to indicate unobtainable value as numeric value 
--infer laterality based on parsing the narrative from pvl_1 above
--some rows are unobtainable but not able to parse laterality

create or replace temp view pvl_unob as
select
deid_person_id,
proc_start_date,
narrative,
anatomy,
specific_anatomy,
null as desc_laterality,
case when rt_unobtainable is not null then 'right' end as narr_right,
case when lt_unobtainable is not null then 'left' end as narr_left,
999 as value
  from pvl_1
where type='measurement'
and unobtainable=1
and anatomy!='arm'

In [0]:
--create dataset for for array=1 not bilateral
--these should be straighforward with 1 pvl value and 1 clear laterality

--stil need to do random assessment to full data

create or replace temp view pvl_array1 as
select 
deid_person_id,
proc_start_date,
narrative,
anatomy,
specific_anatomy,
desc_laterality,
narr_right,
narr_left,
value1 as value
 from pvl_1
where type='measurement'
and float_check=1
and array_length=1
and desc_laterality!='bilateral'
and unobtainable=0

In [0]:
--create dataset for array=1 and bilateral
--these are less straighforward with some indication of bilateral meas but only 1 float value parsed

create or replace temp view pvl_array1b as
select 
deid_person_id,
proc_start_date,
narrative,
anatomy, 
specific_anatomy, 
desc_laterality,
narr_right,
narr_left,  
value1
from pvl_1
where type='measurement'
and float_check=1
and array_length=1
and anatomy!='arm'
and desc_laterality='bilateral'

In [0]:
--generate dataset for array=2
--this is basd on assumption that 1st value is for right and 2nd value is for left

create or replace temp view pvl_array2 as
select 
deid_person_id,
proc_start_date,
narrative,
anatomy, 
specific_anatomy, 
desc_laterality,
'right' as narr_right,
null as narr_left,
value1 as value
from pvl_1
where type='measurement'
and float_check=1
and array_length=2
and anatomy!='arm'

union all

select 
deid_person_id,
proc_start_date,
narrative,
anatomy, 
specific_anatomy, 
desc_laterality,
null as narr_right,
'left' as narr_left,
value2 as value
from pvl_1
where type='measurement'
and float_check=1
and array_length=2
and anatomy!='arm'

In [0]:
select array_length, count(array_length) as freq
from pvl_1
where type='measurement'
and float_check=1
group by array_length


In [0]:

create table nctracs.vascular_pvl_final as
select * from pvl_unob
union all
select * from pvl_array1
union all
select * from pvl_array1b
union all
select * from pvl_array2


In [0]:
select * from pvl_final

In [0]:
select count(deid_person_id) from (
select * from nctracs.vascular_pvl_final
where value<.9
or value>1.2
) X


In [0]:
select sum(freq) as sum from (
select value, count(value) as freq from nctracs.vascular_pvl_final
where value>=.9
and value<=1.2
group by value
) X